In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content/drive/MyDrive/tlqk

In [ ]:
!unzip "features_verts.zip"
!unzip "features_labels.zip"

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from tqdm import tqdm
import warnings
from collections import Counter
import pandas as pd

warnings.filterwarnings('ignore')

# ────────────────────────────────────────────────────
# Configuration
# ────────────────────────────────────────────────────
VERTEX_DIR        = r'/content/drive/MyDrive/tlqk/features_verts'
LABEL_DIR         = r'/content/drive/MyDrive/tlqk/features_labels'
SPIRAL_INDEX_PATH = r'/content/drive/MyDrive/tlqk/spiral_9.npy'
BATCH_SIZE        = 8
LEARNING_RATE     = 1e-3
NUM_EPOCHS        = 80
WEIGHT_DECAY      = 1e-4
NUM_WORKERS       = 0
DEVICE            = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ────────────────────────────────────────────────────
# 개선된 Loss Functions
# ────────────────────────────────────────────────────
class AdaptiveFocalLoss(nn.Module):
    """동적으로 alpha와 gamma를 조정하는 Focal Loss"""
    def __init__(self, alpha=0.25, gamma=2.0, adaptive=True):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.adaptive = adaptive

    def forward(self, logits, targets):
        bce_loss = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        probs = torch.sigmoid(logits)

        # 동적 alpha 계산 (배치 내 positive 비율에 따라)
        if self.adaptive:
            pos_ratio = targets.mean()
            alpha = torch.where(targets == 1, 1 - pos_ratio, pos_ratio)
        else:
            alpha = torch.where(targets == 1, self.alpha, 1 - self.alpha)

        pt = torch.where(targets == 1, probs, 1 - probs)
        focal_weight = alpha * (1 - pt) ** self.gamma
        focal_loss = focal_weight * bce_loss

        return focal_loss.mean()

class HybridLoss(nn.Module):
    """Focal + Dice + Hard Negative Mining"""
    def __init__(self, focal_weight=0.7, dice_weight=0.3, hard_neg_ratio=3.0):
        super().__init__()
        self.focal = AdaptiveFocalLoss(alpha=0.25, gamma=2.0)
        self.focal_weight = focal_weight
        self.dice_weight = dice_weight
        self.hard_neg_ratio = hard_neg_ratio

    def forward(self, logits, targets):
        # Focal Loss
        focal_loss = self.focal(logits, targets)

        # Dice Loss
        probs = torch.sigmoid(logits)
        smooth = 1e-5
        intersection = (probs * targets).sum()
        dice_loss = 1 - (2 * intersection + smooth) / (probs.sum() + targets.sum() + smooth)

        # Hard Negative Mining
        bce_loss = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        pos_mask = targets == 1
        neg_mask = targets == 0

        # Positive samples - 모두 사용
        pos_loss = bce_loss[pos_mask].mean() if pos_mask.sum() > 0 else 0

        # Negative samples - hard negative만 선택
        if neg_mask.sum() > 0:
            neg_losses = bce_loss[neg_mask]
            num_hard_neg = min(int(pos_mask.sum() * self.hard_neg_ratio), neg_mask.sum())
            if num_hard_neg > 0:
                hard_neg_loss = torch.topk(neg_losses, num_hard_neg)[0].mean()
            else:
                hard_neg_loss = neg_losses.mean()
        else:
            hard_neg_loss = 0

        # 최종 loss 조합
        total_loss = (self.focal_weight * focal_loss +
                     self.dice_weight * dice_loss +
                     0.1 * (pos_loss + hard_neg_loss))

        return total_loss

# ────────────────────────────────────────────────────
# 개선된 Dataset with Weighted Sampling
# ────────────────────────────────────────────────────
class ImbalancedCTNoduleDataset(Dataset):
    def __init__(self, verts_dir, labels_dir, bases, augment=False):
        self.verts_dir = verts_dir
        self.labels_dir = labels_dir
        self.bases = bases
        self.augment = augment

        # 각 샘플의 positive 비율 계산 (가중치 샘플링용)
        self.sample_weights = []
        for base in bases:
            labels = np.load(os.path.join(labels_dir, base + '_vertex_labels.npy'))
            pos_ratio = labels.mean()
            # positive가 많은 샘플에 더 높은 가중치
            weight = min(pos_ratio * 10 + 0.1, 2.0)  # 최대 2배까지 가중치
            self.sample_weights.append(weight)

    def __len__(self):
        return len(self.bases)

    def get_sample_weights(self):
        return self.sample_weights

    def __getitem__(self, i):
        base = self.bases[i]
        verts = np.load(os.path.join(self.verts_dir, base + '_features.npy'))
        labels = np.load(os.path.join(self.labels_dir, base + '_vertex_labels.npy'))

        # 간단한 augmentation (결절이 있는 경우)
        if self.augment and labels.mean() > 0.01:
            # 작은 노이즈 추가
            noise = np.random.normal(0, 0.001, verts.shape)
            verts = verts + noise

        return torch.tensor(verts, dtype=torch.float32), torch.tensor(labels, dtype=torch.float32), base

class SimplePointTransformerBlock(nn.Module):
    def __init__(self, dim, num_heads=4, ff_hidden=256, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, num_heads, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        self.ff = nn.Sequential(
            nn.Linear(dim, ff_hidden),
            nn.ReLU(),
            nn.Linear(ff_hidden, dim),
            nn.Dropout(dropout)
        )
    def forward(self, x):
        # x: (B, V, C)
        h = self.norm1(x)
        attn_out, _ = self.attn(h, h, h)
        x = x + attn_out
        h2 = self.norm2(x)
        x = x + self.ff(h2)
        return x

class HybridSpiralNetPointNetTransformer(nn.Module):
    def __init__(self, spiralnet, in_channels, hidden_dim=256, out_dim=1, dropout=0.3,
                 transformer_heads=4, transformer_blocks=1):
        super().__init__()
        self.spiralnet = spiralnet  # 기존 SpiralNet 인스턴스

        # PointNet branch
        self.pointnet_mlp1 = nn.Linear(in_channels, hidden_dim)
        self.pointnet_bn1 = nn.BatchNorm1d(hidden_dim)
        self.pointnet_mlp2 = nn.Linear(hidden_dim, hidden_dim)
        self.pointnet_bn2 = nn.BatchNorm1d(hidden_dim)
        self.pointnet_dropout = nn.Dropout(dropout)

        # Transformer branch (Point Attention)
        self.transformer_blocks = nn.ModuleList([
            SimplePointTransformerBlock(hidden_dim, num_heads=transformer_heads, dropout=dropout)
            for _ in range(transformer_blocks)
        ])

        # 최종적으로 vertex별로 결합 (B, V, hidden*2)
        self.pointnet_head = nn.Linear(hidden_dim * 2, out_dim)

    def forward(self, x, spiral_idx):
        # x: (B, V, C)
        spiral_out = self.spiralnet(x, spiral_idx)  # (B, V)

        B, V, C = x.size()
        # ----- PointNet branch -----
        # Local MLP1
        feat1 = self.pointnet_mlp1(x)  # (B, V, hidden_dim)
        feat1 = feat1.view(B * V, -1)
        feat1 = self.pointnet_bn1(feat1)
        feat1 = F.relu(feat1)
        feat1 = feat1.view(B, V, -1)

        # Local MLP2
        feat2 = self.pointnet_mlp2(feat1)
        feat2 = feat2.view(B * V, -1)
        feat2 = self.pointnet_bn2(feat2)
        feat2 = F.relu(feat2)
        feat2 = feat2.view(B, V, -1)

        point_feat = self.pointnet_dropout(feat2)

        # --- Transformer Attention 추가 ---
        for block in self.transformer_blocks:
            point_feat = block(point_feat)  # (B, V, hidden_dim)

        # Global pooling
        global_feat, _ = torch.max(point_feat, dim=1, keepdim=True)  # (B, 1, hidden_dim)
        global_feat = global_feat.expand(-1, V, -1)  # (B, V, hidden_dim)

        # concat local+global features
        pn_feat = torch.cat([point_feat, global_feat], dim=2)  # (B, V, hidden_dim*2)
        pointnet_out = self.pointnet_head(pn_feat).squeeze(-1)  # (B, V)

        # Soft ensemble (SpiralNet + PointNet)
        out = (spiral_out + pointnet_out) / 2
        return out

# SpiralNet + PointNet 하이브리드 모델 클래스
class HybridSpiralNetPointNet(nn.Module):
    def __init__(self, spiralnet, in_channels, hidden_dim=256, out_dim=1, dropout=0.3):
        super().__init__()
        self.spiralnet = spiralnet  # 기존 SpiralNet 인스턴스

        # PointNet branch: (B, V, C) → (B, V, hidden) → (B, hidden)
        self.pointnet_mlp1 = nn.Linear(in_channels, hidden_dim)
        self.pointnet_bn1 = nn.BatchNorm1d(hidden_dim)
        self.pointnet_mlp2 = nn.Linear(hidden_dim, hidden_dim)
        self.pointnet_bn2 = nn.BatchNorm1d(hidden_dim)
        self.pointnet_dropout = nn.Dropout(dropout)
        # 최종적으로 vertex별로 결합 (B, V, hidden*2)
        self.pointnet_head = nn.Linear(hidden_dim * 2, out_dim)

    def forward(self, x, spiral_idx):
        # x: (B, V, C)
        spiral_out = self.spiralnet(x, spiral_idx)  # (B, V)

        B, V, C = x.size()
        # ----- PointNet branch -----
        # Local MLP1
        feat1 = self.pointnet_mlp1(x)  # (B, V, hidden_dim)
        feat1 = feat1.view(B * V, -1)  # (B*V, hidden_dim)
        feat1 = self.pointnet_bn1(feat1)
        feat1 = F.relu(feat1)
        feat1 = feat1.view(B, V, -1)  # (B, V, hidden_dim)

        # Local MLP2
        feat2 = self.pointnet_mlp2(feat1)
        feat2 = feat2.view(B * V, -1)
        feat2 = self.pointnet_bn2(feat2)
        feat2 = F.relu(feat2)
        feat2 = feat2.view(B, V, -1)  # (B, V, hidden_dim)

        point_feat = self.pointnet_dropout(feat2)  # (B, V, hidden_dim)

        # Global pooling
        global_feat, _ = torch.max(point_feat, dim=1, keepdim=True)  # (B, 1, hidden_dim)
        global_feat = global_feat.expand(-1, V, -1)  # (B, V, hidden_dim)

        # concat local+global features
        pn_feat = torch.cat([point_feat, global_feat], dim=2)  # (B, V, hidden_dim*2)
        pointnet_out = self.pointnet_head(pn_feat).squeeze(-1)  # (B, V)

        # Soft ensemble (SpiralNet + PointNet)
        out = (spiral_out + pointnet_out) / 2
        return out


# ────────────────────────────────────────────────────
# Utils: 개선된 IoU 및 메트릭 계산
# ────────────────────────────────────────────────────
def compute_f1(preds, labels):
    preds = preds.cpu().numpy() if hasattr(preds, 'cpu') else preds
    labels = labels.cpu().numpy() if hasattr(labels, 'cpu') else labels
    tp = np.sum((preds == 1) & (labels == 1))
    fp = np.sum((preds == 1) & (labels == 0))
    fn = np.sum((preds == 0) & (labels == 1))
    precision = tp / (tp + fp + 1e-8)
    recall = tp / (tp + fn + 1e-8)
    f1 = 2 * precision * recall / (precision + recall + 1e-8)
    return f1, precision, recall

def compute_iou(preds, labels):
    preds = torch.as_tensor(preds).float().view(-1)
    labels = torch.as_tensor(labels).float().view(-1)
    intersection = ((preds == 1) & (labels == 1)).sum().float()
    union = ((preds == 1) | (labels == 1)).sum().float()
    if union == 0:
        return torch.tensor(1.0) if intersection == 0 else torch.tensor(0.0)
    return intersection / union

def find_best_threshold_advanced(all_probs, all_labels, metric='f1'):
    """F1, IoU, Balanced Accuracy 등 다양한 메트릭으로 최적 임계값 찾기"""
    thresholds = np.linspace(0.01, 0.99, 99)
    best_thresh, best_score = 0.5, 0.0

    for t in thresholds:
        preds = (all_probs > t).float()

        if metric == 'f1':
            f1, _, _ = compute_f1(preds, all_labels)
            score = f1
        elif metric == 'iou':
            score = compute_iou(preds, all_labels).item()
        elif metric == 'balanced_acc':
            tp = ((preds == 1) & (all_labels == 1)).sum().float()
            tn = ((preds == 0) & (all_labels == 0)).sum().float()
            fp = ((preds == 1) & (all_labels == 0)).sum().float()
            fn = ((preds == 0) & (all_labels == 1)).sum().float()
            sensitivity = tp / (tp + fn + 1e-8)
            specificity = tn / (tn + fp + 1e-8)
            score = (sensitivity + specificity) / 2

        if score > best_score:
            best_score, best_thresh = score, t

    return best_thresh, best_score

class HybridSpiralNetMLP(nn.Module):
    def __init__(self, spiralnet, mlp_hidden=256, num_classes=1):
        super().__init__()
        self.spiralnet = spiralnet  # 기존 SpiralNet 구조
        in_dim = spiralnet.in_channels
        # MLP branch (예: 3-layer)
        self.mlp = nn.Sequential(
            nn.Linear(in_dim, mlp_hidden),
            nn.ReLU(),
            nn.Linear(mlp_hidden, mlp_hidden//2),
            nn.ReLU(),
            nn.Linear(mlp_hidden//2, num_classes)
        )
        # 최종 head (concat된 feature로)
        self.head = nn.Sequential(
            nn.Linear(2 * num_classes, num_classes)  # spiral + mlp output concat
        )
    def forward(self, x, spiral_idx):
        # x: (B, V, C)
        # SpiralNet output (B, V)
        out_spiral = self.spiralnet(x, spiral_idx)
        # MLP output (B, V)
        out_mlp = self.mlp(x)
        # concat outputs
        out = torch.cat([out_spiral.unsqueeze(-1), out_mlp], dim=-1)
        out = self.head(out).squeeze(-1)
        return out

# ────────────────────────────────────────────────────
# SpiralNet Model (기존과 동일)
# ────────────────────────────────────────────────────
class SpiralConv(nn.Module):
    def __init__(self, in_channels, out_channels, spiral_len: int):
        super().__init__()
        self.spiral_len = spiral_len
        self.conv = nn.Conv1d(in_channels * spiral_len, out_channels, kernel_size=1)
        self.bn = nn.BatchNorm1d(out_channels)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x, spiral_idx):
        B, C, V = x.size()
        L = spiral_idx.size(1)
        idx = spiral_idx.unsqueeze(0).unsqueeze(0).expand(B, C, V, L)
        x_expand = x.unsqueeze(-1).expand(B, C, V, L)
        neigh = torch.gather(x_expand, 2, idx)
        neigh = neigh.reshape(B, C * L, V)
        out = self.conv(neigh)
        out = self.bn(out)
        return self.relu(out)

class SpiralBlock(nn.Module):
    def __init__(self, in_channels, out_channels, spiral_index, dropout=0.0):
        super().__init__()
        spiral_len = spiral_index.shape[1]
        self.spiral = SpiralConv(in_channels, out_channels, spiral_len)
        self.use_res = (in_channels == out_channels)
        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

    def forward(self, x, spiral_idx):
        out = self.spiral(x, spiral_idx)
        out = self.dropout(out)
        if self.use_res:
            return out + x
        else:
            return out

class SEBlock(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        w = self.pool(x).squeeze(-1)
        w = self.fc(w).unsqueeze(-1)
        return x * w

class ImprovedSpiralNet(nn.Module):
    """개선된 SpiralNet with Multi-scale features"""
    def __init__(self, in_channels, hidden_channels, out_channels,
                 num_blocks, spiral_indices=None, dropout=0.0, use_se=True):
        super().__init__()
        self.spiral_indices = spiral_indices
        num_blocks = len(spiral_indices)
        self.use_se = use_se

        self.encoder = nn.ModuleList()
        for i in range(num_blocks):
            in_c = in_channels if i == 0 else hidden_channels
            out_c = hidden_channels
            spiral_index = spiral_indices[i]
            self.encoder.append(
                SpiralBlock(in_c, out_c, spiral_index, dropout=dropout)
            )

        if use_se:
            self.se_blocks = nn.ModuleList([
                SEBlock(hidden_channels) for _ in range(num_blocks)
            ])
        else:
            self.se_blocks = [nn.Identity()] * num_blocks

        self.skip_conns = nn.ModuleList()
        self.decoder = nn.ModuleList()
        for i in range(num_blocks - 1):
            self.skip_conns.append(nn.Conv1d(hidden_channels * 2, hidden_channels, kernel_size=1))
            spiral_index_dec = spiral_indices[num_blocks - 2 - i]
            self.decoder.append(
                SpiralBlock(hidden_channels, hidden_channels, spiral_index_dec, dropout=dropout)
            )

        # Multi-scale classifier
        self.classifier = nn.Sequential(
            nn.Conv1d(hidden_channels, hidden_channels // 2, kernel_size=1),
            nn.BatchNorm1d(hidden_channels // 2),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Conv1d(hidden_channels // 2, hidden_channels // 4, kernel_size=1),
            nn.BatchNorm1d(hidden_channels // 4),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Conv1d(hidden_channels // 4, out_channels, kernel_size=1)
        )

    def forward(self, verts, indices):
        x = verts.permute(0, 2, 1)
        features = []

        for block, idx, se in zip(self.encoder, self.spiral_indices, self.se_blocks):
            x = block(x, idx)
            x = se(x)
            features.append(x)

        for skip_conv, dec_block, idx, feat in zip(
            self.skip_conns,
            self.decoder,
            reversed(self.spiral_indices[:-1]),
            reversed(features[:-1])
        ):
            x = torch.cat([x, feat], dim=1)
            x = skip_conv(x)
            x = dec_block(x, idx)

        out = self.classifier(x)
        return out.squeeze(1)

# ────────────────────────────────────────────────────
# 개선된 Training & Validation
# ────────────────────────────────────────────────────
def train_epoch_improved(model, loader, criterion, optimizer, spiral_idx, epoch=None, total_epochs=None):
    model.train()
    total_loss = 0
    all_probs, all_lbls = [], []
    pos_samples, neg_samples = 0, 0

    bar = tqdm(loader, desc=f"Train [{epoch}/{total_epochs}]", leave=False, ncols=100)
    for step, (verts, labels, base) in enumerate(bar):
        verts, labels = verts.to(DEVICE), labels.to(DEVICE)
        if labels.dim() == 3 and labels.shape[-1] == 1:
            labels = labels.squeeze(-1)

        optimizer.zero_grad()
        out = model(verts, spiral_idx)
        loss = criterion(out, labels)
        loss.backward()

        # Gradient clipping for stability
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        total_loss += loss.item()

        probs = torch.sigmoid(out).detach().cpu().view(-1)
        labels_flat = labels.detach().cpu().view(-1)

        all_probs.append(probs)
        all_lbls.append(labels_flat)

        pos_samples += (labels_flat == 1).sum().item()
        neg_samples += (labels_flat == 0).sum().item()

        bar.set_postfix(loss=f"{loss.item():.4f}", pos=pos_samples, neg=neg_samples)

    all_probs = torch.cat(all_probs)
    all_lbls = torch.cat(all_lbls)

    # Training epoch 통계
    print(f"[Train Epoch {epoch}] Pos: {pos_samples}, Neg: {neg_samples}, Ratio: 1:{neg_samples/max(pos_samples,1):.1f}")

    return total_loss / len(loader)

def validate_epoch_improved(model, loader, criterion, spiral_idx, epoch=None, total_epochs=None):
    model.eval()
    total_loss = 0
    all_probs, all_lbls = [], []
    sample_metrics = []

    bar = tqdm(loader, desc=f"Valid [{epoch}/{total_epochs}]", leave=False, ncols=100)
    with torch.no_grad():
        for i, (verts, labels, base) in enumerate(bar):
            verts, labels = verts.to(DEVICE), labels.to(DEVICE).squeeze(-1)
            out = model(verts, spiral_idx)
            loss = criterion(out, labels)
            total_loss += loss.item()

            probs = torch.sigmoid(out).view(-1)
            labels_flat = labels.view(-1)

            all_probs.append(probs)
            all_lbls.append(labels_flat)

            # 샘플별 메트릭 (임계값 0.5 기준)
            preds_bin = (probs > 0.5).float()
            iou = compute_iou(preds_bin, labels_flat).item()
            f1, precision, recall = compute_f1(preds_bin, labels_flat)
            acc = (preds_bin == labels_flat).float().mean().item()

            sample_metrics.append({
                "base": base[0] if isinstance(base, (list, tuple)) else base,
                "IoU": iou, "F1": f1, "Precision": precision,
                "Recall": recall, "Acc": acc,
                "pos_ratio": labels_flat.mean().item()
            })

    all_probs = torch.cat(all_probs)
    all_lbls = torch.cat(all_lbls)

    # 다양한 메트릭으로 최적 임계값 찾기
    best_t_f1, best_f1 = find_best_threshold_advanced(all_probs, all_lbls, 'f1')
    best_t_iou, best_iou = find_best_threshold_advanced(all_probs, all_lbls, 'iou')
    best_t_bal, best_bal = find_best_threshold_advanced(all_probs, all_lbls, 'balanced_acc')

    # F1 기준 최종 메트릭 계산
    preds_final = (all_probs > best_t_f1).float()
    final_f1, final_precision, final_recall = compute_f1(preds_final, all_lbls)
    final_acc = (preds_final == all_lbls).float().mean().item()
    final_iou = compute_iou(preds_final, all_lbls).item()

    print(f"[Valid] F1={final_f1:.4f}(t={best_t_f1:.2f}) | IoU={final_iou:.4f}(t={best_t_iou:.2f}) | "
          f"Acc={final_acc:.4f} | P={final_precision:.4f} | R={final_recall:.4f}")

    return total_loss/len(loader), final_f1, best_t_f1, sample_metrics


# ────────────────────────────────────────────────────
# Main Runner
# ────────────────────────────────────────────────────
def run_improved(resume_path=None):
    print("🚀 개선된 불균형 처리 SpiralNet 학습 시작")

    spiral_all = np.load(SPIRAL_INDEX_PATH, allow_pickle=True)
    num_blocks = 4
    spiral_idx = [torch.from_numpy(spiral_all).long().to(DEVICE) for _ in range(num_blocks)]

    files = sorted([f for f in os.listdir(VERTEX_DIR) if f.endswith('_features.npy')])
    bases = [v[:-13] for v in files]
    bases = [b for b in bases if os.path.exists(os.path.join(LABEL_DIR, b+'_vertex_labels.npy'))]


    example_feat = np.load(os.path.join(VERTEX_DIR, bases[0] + '_features.npy'))
    feature_dim = example_feat.shape[1] if example_feat.ndim == 2 else example_feat.shape[-1]

    # 전체 데이터 불균형 분석
    labels_all = np.concatenate([np.load(os.path.join(LABEL_DIR, b+'_vertex_labels.npy')).ravel() for b in bases])

    num_pos = (labels_all == 1).sum()
    num_neg = (labels_all == 0).sum()
    pos_weight_value = num_neg / num_pos * 0.7
    print(f"pos_weight: {pos_weight_value:.2f}")
    pos_weight = torch.tensor([pos_weight_value]).to(DEVICE)




    unique, counts = np.unique(labels_all, return_counts=True)
    print(f"📊 전체 레이블 분포: {dict(zip(unique, counts))}")
    print(f"📊 불균형 비율: 1:{counts[0]/counts[1]:.1f} (결절:비결절)")

    # Train/Val 분할
    train_b, val_b = train_test_split(bases, test_size=0.2, random_state=42, stratify=None)

    # 개선된 Dataset 생성
    train_dataset = ImbalancedCTNoduleDataset(VERTEX_DIR, LABEL_DIR, train_b, augment=True)
    val_dataset = ImbalancedCTNoduleDataset(VERTEX_DIR, LABEL_DIR, val_b, augment=False)

    # WeightedRandomSampler 적용
    sample_weights = train_dataset.get_sample_weights()
    sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=NUM_WORKERS)
    val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, num_workers=NUM_WORKERS)

    print(f"📈 Train: {len(train_b)}, Val: {len(val_b)} samples")

    # 개선된 모델 및 Loss
#     model = ImprovedSpiralNet(
#         in_channels=3, hidden_channels=256, out_channels=1, num_blocks=8,
#         spiral_indices=spiral_idx, dropout=0.3, use_se=True
#     ).to(DEVICE)

    spiralnet = ImprovedSpiralNet(
        in_channels=feature_dim,  # 예: 3 (x,y,z) 또는 7
        hidden_channels=256,
        out_channels=1,
        num_blocks=4,
        spiral_indices=spiral_idx,
        dropout=0.3,
        use_se=True
    ).to(DEVICE)


    model = HybridSpiralNetPointNetTransformer(
        spiralnet=spiralnet,
        in_channels=feature_dim,
        hidden_dim=256,
        out_dim=1,
        dropout=0.3,
        transformer_heads=4,
        transformer_blocks=1,   # 1~2 블록 권장 (메모리↑시 2까지)
    ).to(DEVICE)

    #criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    #criterion = HybridLoss(focal_weight=0.7, dice_weight=0.3, hard_neg_ratio=3.0)
    criterion = AdaptiveFocalLoss(alpha=0.25, gamma=2.0, adaptive=True)

    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

    # 학습률 스케줄러 (Warm-up + Cosine)
    def lr_lambda(epoch):
        if epoch < 5:  # Warm-up
            return epoch / 5
        else:
            return 0.5 * (1 + np.cos(np.pi * (epoch - 5) / (NUM_EPOCHS - 5)))

    scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    best_f1 = 0.0
    patience = 10
    patience_counter = 0
    start_epoch = 1

    # resume 기능 추가!
    if resume_path is not None and os.path.exists(resume_path):
        print(f"🔄 체크포인트 불러오는 중: {resume_path}")
        checkpoint = torch.load(resume_path, map_location=DEVICE, weights_only=False)
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        best_f1 = checkpoint.get('best_f1', 0.0)
        start_epoch = checkpoint.get('epoch', 1) + 1  # 다음 epoch부터 시작
        print(f"  ↪️  {start_epoch-1}번째 epoch에서 이어서 시작 (Best F1={best_f1:.4f})")

    print(f"{'Ep':>3} {'TrainL':>8} {'ValL':>8} {'F1':>8} {'Best':>8} {'Thresh':>7} {'LR':>8}")
    print('-' * 65)

    for epoch in range(start_epoch, NUM_EPOCHS + 1):
        train_loss = train_epoch_improved(model, train_loader, criterion, optimizer, spiral_idx, epoch, NUM_EPOCHS)
        val_loss, val_f1, val_thresh, sample_metrics = validate_epoch_improved(
            model, val_loader, criterion, spiral_idx, epoch, NUM_EPOCHS
        )

        scheduler.step()
        lr = optimizer.param_groups[0]['lr']

        # Best model 저장
        if val_f1 > best_f1:
            best_f1 = val_f1
            torch.save({
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'epoch': epoch,
                'best_f1': best_f1,
                'threshold': val_thresh
            }, 'tlqkf.pth')
            patience_counter = 0
        else:
            patience_counter += 1

        print(f"{epoch:3d} {train_loss:8.4f} {val_loss:8.4f} {val_f1:8.4f} {best_f1:8.4f} {val_thresh:7.2f} {lr:8.2e}")

        # Early stopping
        if patience_counter >= patience:
            print(f"⏰ Early stopping at epoch {epoch}")
            break

    # 최종 결과 분석
    print(f"\n🎯 최종 결과: Best F1 = {best_f1:.4f}")

    # # 최악 성능 샘플 분석
    # df_metrics = pd.DataFrame(sample_metrics)
    # worst_samples = df_metrics.nsmallest(5, 'F1')
    # print("\n📉 F1 점수가 낮은 상위 5개 샘플:")
    # print(worst_samples[['base', 'F1', 'IoU', 'Precision', 'Recall', 'pos_ratio']].round(3))

    # 결절 비율별 성능 분석
    # df_metrics['pos_ratio_bin'] = pd.cut(df_metrics['pos_ratio'],
    #                                     bins=[0, 0.01, 0.05, 0.1, 1.0],
    #                                     labels=['<1%', '1-5%', '5-10%', '>10%'])
    # performance_by_ratio = df_metrics.groupby('pos_ratio_bin')[['F1', 'IoU', 'Precision', 'Recall']].mean()
    # print("\n📊 결절 비율별 평균 성능:")
    # print(performance_by_ratio.round(3))

if __name__ == '__main__':
    # 이어서 학습하고 싶을 때는 파일명 넣어서!
    #run_improved(resume_path='/content/tlqkf.pth')
    run_improved()


In [ ]:
import re
import matplotlib.pyplot as plt

text = """
  1   0.0199   0.0104   0.0609   0.0609    0.51 2.00e-04
  2   0.0123   0.0069   0.3566   0.3566    0.75 4.00e-04
  3   0.0103   0.0073   0.3514   0.3566    0.69 6.00e-04
  4   0.0096   0.0060   0.4135   0.4135    0.72 8.00e-04
  5   0.0094   0.0058   0.4359   0.4359    0.76 1.00e-03
  6   0.0095   0.0055   0.4535   0.4535    0.73 1.00e-03
  7   0.0078   0.0053   0.5342   0.5342    0.76 9.98e-04
  8   0.0069   0.0059   0.5380   0.5380    0.79 9.96e-04
  9   0.0066   0.0051   0.4676   0.5380    0.72 9.93e-04
 10   0.0055   0.0042   0.5891   0.5891    0.77 9.89e-04
 11   0.0058   0.0035   0.5844   0.5891    0.81 9.84e-04
 12   0.0051   0.0067   0.5073   0.5891    0.74 9.79e-04
 13   0.0047   0.0038   0.6334   0.6334    0.79 9.72e-04
 14   0.0044   0.0050   0.5986   0.6334    0.81 9.65e-04
 15   0.0041   0.0074   0.5832   0.6334    0.79 9.57e-04
 16   0.0042   0.0039   0.5761   0.6334    0.77 9.48e-04
 17   0.0039   0.0039   0.6397   0.6397    0.78 9.38e-04
 18   0.0038   0.0034   0.6734   0.6734    0.76 9.28e-04
 19   0.0034   0.0027   0.6867   0.6867    0.80 9.16e-04
 20   0.0031   0.0028   0.6997   0.6997    0.74 9.05e-04
 21   0.0033   0.0026   0.7038   0.7038    0.76 8.92e-04
 22   0.0029   0.0025   0.6973   0.7038    0.79 8.78e-04
 23   0.0028   0.0020   0.7316   0.7316    0.77 8.64e-04
 24   0.0031   0.0022   0.7208   0.7316    0.80 8.50e-04
 25   0.0028   0.0020   0.7326   0.7326    0.80 8.35e-04
 26   0.0026   0.0024   0.7008   0.7326    0.79 8.19e-04
 27   0.0024   0.0018   0.7376   0.7376    0.74 8.02e-04
 28   0.0023   0.0025   0.7130   0.7376    0.72 7.85e-04
 29   0.0023   0.0015   0.7681   0.7681    0.77 7.68e-04
 30   0.0022   0.0018   0.7558   0.7681    0.73 7.50e-04
 31   0.0021   0.0018   0.7500   0.7681    0.78 7.32e-04
 32   0.0020   0.0018   0.7469   0.7681    0.73 7.13e-04
 33   0.0021   0.0019   0.7434   0.7681    0.72 6.94e-04
 34   0.0019   0.0019   0.7585   0.7681    0.72 6.74e-04
 35   0.0018   0.0016   0.7470   0.7681    0.74 6.55e-04
 36   0.0018   0.0013   0.7978   0.7978    0.70 6.34e-04
 37   0.0017   0.0017   0.7582   0.7978    0.69 6.14e-04
 38   0.0018   0.0015   0.7784   0.7978    0.68 5.94e-04
 39   0.0016   0.0015   0.7821   0.7978    0.65 5.73e-04
 40   0.0015   0.0018   0.7630   0.7978    0.69 5.52e-04
 41   0.0015   0.0016   0.7845   0.7978    0.64 5.31e-04
 42   0.0015   0.0022   0.7583   0.7978    0.63 5.10e-04
 43   0.0014   0.0017   0.7852   0.7978    0.61 4.90e-04
 44   0.0015   0.0021   0.7677   0.7978    0.61 4.69e-04
 45   0.0014   0.0017   0.7852   0.7978    0.60 4.48e-04
 46   0.0014   0.0018   0.7851   0.7978    0.61 4.27e-04
"""

# 정규식으로 라인 파싱
pattern = re.compile(r"^\s*(\d+)\s+([\d\.]+)\s+([\d\.]+)\s+([\d\.]+)\s+([\d\.]+)\s+([\d\.]+)\s+([\d\.eE+-]+)", re.MULTILINE)
results = pattern.findall(text)

# 값 분리 및 변환
epochs, train_losses, val_losses, f1s, bests, threshs, lrs = [], [], [], [], [], [], []
for r in results:
    epochs.append(int(r[0]))
    train_losses.append(float(r[1]))
    val_losses.append(float(r[2]))
    f1s.append(float(r[3]))
    bests.append(float(r[4]))
    threshs.append(float(r[5]))
    lrs.append(float(r[6]))

# 그래프 그리기
plt.figure(figsize=(14,7))
plt.subplot(2,1,1)
plt.plot(epochs, train_losses, label='Train Loss', marker='o', alpha=0.7)
plt.plot(epochs, val_losses, label='Val Loss', marker='o', alpha=0.7)
plt.ylabel('Loss')
plt.legend()
plt.title('Training/Validation Loss per Epoch')

plt.subplot(2,1,2)
plt.plot(epochs, f1s, label='F1', marker='o')
plt.plot(epochs, bests, label='Best F1', marker='s')
plt.xlabel('Epoch')
plt.ylabel('F1 Score')
plt.legend()
plt.title('F1 & Best F1 per Epoch')

plt.tight_layout()
plt.show()
